# lda_dcabp_evolution

Evolution of LDA behavioral patterns over time per patient.

1. Load the **day-type sequences**, the trained **LDA model** and the **dictionary**.
2. Attach the clinical dates / event info.
3. Build the **monthly predominant topic** table (30-day blocks).
4. Global plots:
   - **Predominant topic per month** heatmap (6 topics).
   - **Hypertopic per month** heatmap (favorable vs unfavorable groups).

In [ ]:
import ast
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from gensim import corpora
from gensim.models import LdaModel

def find_project_root() -> Path:
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    for p in (cwd, *cwd.parents):
        if (p / "models").is_dir() and (p / "data").is_dir():
            return p
    return cwd


def rel_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


PROJECT_ROOT = find_project_root()

# Trained LDA artifacts
LDA_MODEL_PATH = PROJECT_ROOT / "models/lda/LDA_model_vdec25.gensim"
DICTIONARY_PATH = PROJECT_ROOT / "models/lda/dictionary_LDA_vdec25.dict"

# eB2 table with day-type sequences (id, embedding_ids) + clinical dates/event
DF_DAY_CSV = PROJECT_ROOT / "data/raw/PD_cutoff_dic_2025s_eB2_Topics_a0_6topics100000.csv"

# Date / event columns to read from DF_DAY_CSV
DATE_COLS = [
    "Fecha_entrada_HDM", "Evento PD", "Tiempo Obs Final",
    "diferencia", "Evento PD diff", "early PD", "primer_registro_EB2",
]

# Reuse the monthly-topics helper from the scripts package
SCRIPTS_LDA = PROJECT_ROOT / "scripts/03_analysis/lda"
if str(SCRIPTS_LDA) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_LDA))

WINDOW_SIZE = 30
MIN_EMBEDDINGS = 15

TABLES_DIR = PROJECT_ROOT / "results/lda/tables"
FIG_DIR = PROJECT_ROOT / "results/lda/figures"



## 1 · Load embeddings, LDA model and dictionary

In [ ]:
lda_model = LdaModel.load(str(LDA_MODEL_PATH))
dictionary = corpora.Dictionary.load(str(DICTIONARY_PATH))
NUM_TOPICS = lda_model.num_topics
print(f"LDA topics: {NUM_TOPICS} | dictionary size: {len(dictionary)}")

df = pd.read_csv(DF_DAY_CSV)
df = df.rename(columns={"user_id": "id"})
df["embedding_ids"] = df["embedding_ids"].apply(
    lambda v: ast.literal_eval(v) if isinstance(v, str) else v
)
df["id"] = pd.to_numeric(df["id"], errors="coerce").astype("Int64")
print("Patients:", len(df))
df.head()

## 2 · Select day-type sequences + clinical dates / event



In [ ]:
missing = [c for c in DATE_COLS if c not in df.columns]
if "Evento PD" in missing or "Evento PD diff" in missing:
    raise KeyError(f"eB2 CSV is missing required columns: {missing}")

keep = ["id", "embedding_ids"] + [c for c in DATE_COLS if c in df.columns]
df_day = df[keep].copy()
print("df_day:", df_day.shape)
df_day[["id", "Evento PD", "Evento PD diff"]].head()

## 3 · Monthly predominant topic table (30-day blocks)

In [ ]:
from monthly_topics_utils import (
    build_monthly_predominant_table,
    sort_patients_for_heatmap,
    topic_month_columns,
)

ordered_df = build_monthly_predominant_table(
    df_day, lda_model, dictionary,
    window_size=WINDOW_SIZE, min_embeddings=MIN_EMBEDDINGS,
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
monthly_csv = TABLES_DIR / "monthly_predominant_topics.csv"
ordered_df.to_csv(monthly_csv, index=False)
print("Saved:", rel_path(monthly_csv))
print("Patients in table:", len(ordered_df))
ordered_df.head()

## 4 · Global plot 1 — Predominant topic per month (6 topics)

In [ ]:
# Discrete colors per topic
HEATMAP_TOPIC_COLORS = {
    0: "#32cd32", 1: "#228b22", 2: "#f0e68c",
    3: "#b22222", 4: "#ffd700", 5: "#ff6347",
}

topic_cols = topic_month_columns(ordered_df)
ordered_sorted = sort_patients_for_heatmap(ordered_df, topic_cols)

plot_df = (
    ordered_sorted[["id"] + topic_cols]
    .set_index("id")
    .replace({pd.NA: np.nan})
    .apply(pd.to_numeric, errors="coerce")
)

color_list = [HEATMAP_TOPIC_COLORS[i] for i in sorted(HEATMAP_TOPIC_COLORS)]
cmap = ListedColormap(color_list)

fig, ax = plt.subplots(figsize=(12, len(plot_df) * 0.15 + 1))
sns.heatmap(
    plot_df, cmap=cmap, vmin=0, vmax=len(color_list) - 1,
    linewidths=0.5, linecolor="lightgray",
    cbar_kws={"label": "Predominant Topic"},
    mask=plot_df.isna(), ax=ax,
)

x_axis = [re.findall(r"\d+", c)[-1] for c in topic_cols]
ax.set_xticklabels(x_axis)

for i, row in ordered_sorted.iterrows():
    if row["Event"] == 1 and not pd.isna(row.get("t_evento_eb2")):
        mes_evento = int(row["t_evento_eb2"] // 30)
        if 0 <= mes_evento < len(topic_cols):
            ax.add_patch(plt.Rectangle((mes_evento, i), 1, 1, fill=False,
                                       edgecolor="black", linewidth=2))

n_no_pd = len(ordered_sorted[ordered_sorted["Event"] == 0])
ax.axhline(n_no_pd, color="black", linewidth=2.5)

ax.set_title(f"Predominant topic per month ({NUM_TOPICS} topics)")
ax.set_xlabel("Month")
ax.set_ylabel("Patient ID")
plt.tight_layout()

FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "predominant_topics_per_month.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "predominant_topics_per_month.svg", bbox_inches="tight")
print("Saved:", rel_path(FIG_DIR / "predominant_topics_per_month.svg"))
plt.show()

## 5 · Global plot 2 — Hypertopic per month (favorable vs unfavorable)

In [ ]:
# Collapse the 6 topics into 2 groups (0 = favorable, 1 = unfavorable)
group_map = {0: 0, 1: 0, 2: 0, 3: 1, 4: 1, 5: 1}
plot_df_grouped = plot_df.replace(group_map)

group_colors = ["#2ca02c", "#d62728"]
cmap_groups = ListedColormap(group_colors)

fig, ax = plt.subplots(figsize=(10, len(plot_df_grouped) * 0.15 + 1))
sns.heatmap(
    plot_df_grouped, cmap=cmap_groups, vmin=0, vmax=1,
    linewidths=0.5, linecolor="lightgray",
    mask=plot_df_grouped.isna(),
    xticklabels=x_axis,
    cbar_kws={"label": "Topic Group\n0 = Favorable, 1 = Unfavorable"},
    ax=ax,
)

for i, row in ordered_sorted.iterrows():
    if row["Event"] == 1 and not pd.isna(row.get("t_evento_eb2")):
        mes_evento = int(row["t_evento_eb2"] // 30)
        if 0 <= mes_evento < len(topic_cols):
            ax.add_patch(plt.Rectangle((mes_evento, i), 1, 1, fill=False,
                                       edgecolor="black", linewidth=2, linestyle="--"))

n_no_pd = len(ordered_sorted[ordered_sorted["Event"] == 0])
ax.axhline(n_no_pd, color="black", linewidth=2.5)

ax.set_title(f"Predominant hypertopic per month (PD-HT: 3, 4, 5; Topics: {NUM_TOPICS})")
ax.set_xlabel("Month")
ax.set_ylabel("Patient ID")
plt.tight_layout()

fig.savefig(FIG_DIR / "hypertopics_per_month.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "hypertopics_per_month.svg", bbox_inches="tight")
print("Saved:", rel_path(FIG_DIR / "hypertopics_per_month.svg"))
plt.show()